# PDF Comment Tools in Google Colab

This notebook installs `py-pdf_comment_tools` from GitHub and gives you a simple single-PDF workflow for:

- `highlight-keywords`: upload one PDF and one keywords CSV, then download the annotated PDF and summary CSV
- `extract-comments`: upload one annotated PDF, then download the extracted CSV


## 1. Setup

Run the next cell once per Colab session. It clones the repo into `/content/py-pdf_comment_tools`, installs the package, and prepares input/output folders.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/dariru3/py-pdf_comment_tools.git"
REPO_REF = "colab-support"
REPO_DIR = Path("/content/py-pdf_comment_tools")
WORK_DIR = Path("/content/work")
INPUT_DIR = WORK_DIR / "inputs"
OUTPUT_DIR = WORK_DIR / "outputs"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)

for path in (INPUT_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f"Repo ready at {REPO_DIR} from {REPO_REF}")
print(f"Inputs: {INPUT_DIR}")
print(f"Outputs: {OUTPUT_DIR}")


In [ ]:
from google.colab import files
from pdf_comment_tools import (
    DEFAULT_COMMENTS_OUTPUT,
    DEFAULT_SUMMARY_NAME,
    default_annotated_pdf_path,
    extract_comment_rows,
    highlight_keywords,
    load_keywords,
    parse_pages,
    run_extract_comments,
)

def reset_directory(path):
    path.mkdir(parents=True, exist_ok=True)
    for child in path.iterdir():
        if child.is_file():
            child.unlink()

def upload_to(directory):
    directory.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    saved = []
    for name, data in uploaded.items():
        target = directory / name
        target.write_bytes(data)
        saved.append(target)
    return saved


## 2. Highlight keywords and add comments

Upload one PDF and one CSV with columns `keyword`, `comment`, and optional `color`.

In [ ]:
reset_directory(INPUT_DIR)
reset_directory(OUTPUT_DIR)

print("Upload a single PDF and a keywords CSV when prompted.")
uploaded_paths = upload_to(INPUT_DIR)
print("Uploaded:")
for path in uploaded_paths:
    print(" -", path.name)

pdf_path = next(path for path in uploaded_paths if path.suffix.lower() == ".pdf")
keywords_csv = next(path for path in uploaded_paths if path.suffix.lower() == ".csv")
pages_arg = None  # Example: "1,3-5"


In [ ]:
pages = parse_pages(pages_arg)
keywords = load_keywords(str(keywords_csv))
summary_path = OUTPUT_DIR / DEFAULT_SUMMARY_NAME

highlight_keywords(
    pdf_paths=[pdf_path],
    keywords=keywords,
    pages=pages,
    output_dir=OUTPUT_DIR,
    summary_path=summary_path,
)

annotated_pdf = default_annotated_pdf_path(pdf_path, OUTPUT_DIR)
print(f"Annotated PDF: {annotated_pdf}")
print(f"Summary CSV: {summary_path}")


In [ ]:
files.download(str(annotated_pdf))
files.download(str(summary_path))


## 3. Extract comments from one PDF

Upload one PDF that already contains comments/highlights. The output is `annotation_comments.csv`.

In [ ]:
reset_directory(INPUT_DIR)
reset_directory(OUTPUT_DIR)

print("Upload one annotated PDF when prompted.")
uploaded_paths = upload_to(INPUT_DIR)
pdf_path = next(path for path in uploaded_paths if path.suffix.lower() == ".pdf")
pages_arg = None  # Example: "2-4"
print(f"Using {pdf_path.name}")


In [ ]:
pages = parse_pages(pages_arg)
output_csv = OUTPUT_DIR / DEFAULT_COMMENTS_OUTPUT
run_extract_comments(pdf_path, pages, output_csv)
rows = extract_comment_rows(pdf_path, pages)
print(f"Wrote {output_csv}")
print(f"Extracted {len(rows)} rows")


In [ ]:
files.download(str(output_csv))
